### Linear regression/diagnostics/cross validation

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn import linear_model
from sklearn.linear_model import LinearRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multiclass import OneVsOneClassifier
from sklearn.metrics import ConfusionMatrixDisplay

import statsmodels.api as sm
import seaborn as sns

from scipy import stats

In [3]:
data = pd.read_csv("dataset_2191_sleep.csv")
#clean data: drop question mark values
data = data.replace("?", np.nan)
data = data.dropna(how='any')
data.info()

<class 'pandas.DataFrame'>
Index: 51 entries, 0 to 60
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   body_weight           51 non-null     float64
 1   brain_weight          51 non-null     float64
 2   max_life_span         51 non-null     str    
 3   gestation_time        51 non-null     str    
 4   predation_index       51 non-null     int64  
 5   sleep_exposure_index  51 non-null     int64  
 6   danger_index          51 non-null     int64  
 7   total_sleep           51 non-null     str    
dtypes: float64(2), int64(3), str(3)
memory usage: 3.6 KB


In [4]:
#convert strings to floats
data["max_life_span"] = data["max_life_span"].astype(float)
data["total_sleep"] = data["total_sleep"].astype(float)
data.info()

<class 'pandas.DataFrame'>
Index: 51 entries, 0 to 60
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   body_weight           51 non-null     float64
 1   brain_weight          51 non-null     float64
 2   max_life_span         51 non-null     float64
 3   gestation_time        51 non-null     str    
 4   predation_index       51 non-null     int64  
 5   sleep_exposure_index  51 non-null     int64  
 6   danger_index          51 non-null     int64  
 7   total_sleep           51 non-null     float64
dtypes: float64(4), int64(3), str(1)
memory usage: 3.6 KB


In [5]:
#define dependent and independent variables for later use
max_life_span = data["max_life_span"]
sleep_time = data["total_sleep"]

In [6]:
#split data

X_train, X_test, y_train, y_test = train_test_split(max_life_span, sleep_time, test_size=0.3, random_state=42)
#convert everything to numpy
X_train = X_train.to_numpy()
y_test = y_test.to_numpy()
y_train = y_train.to_numpy()
X_test = X_test.to_numpy()

#fix dimensions to make this work
X_test = X_test.reshape(-1, 1)
X_train = X_train.reshape(-1, 1)
y_train = y_train.reshape(-1, 1)

In [14]:
# Make sure dependent variables are 1D
y_train = y_train.ravel()
y_test = y_test.ravel()

In [7]:
#since X_train is only 35 -> chose k = 2 so that each time the model runs there is at least some meaningful amount of data
k = 2
kf = KFold(n_splits=k,shuffle=True,random_state=42)

In [8]:
#train model 
mse_ols_results = []


for (train, val) in kf.split(X_train):
    x_t, y_t = X_train[train], y_train[train]
    x_v, y_v = X_train[val], y_train[val]
    
    model1 = LinearRegression()
    model1.fit(x_t, y_t)

    # Predict on the validation observations
    predictions = model1.predict(x_v)
    
    # Calculate validation MSE
    mse = mean_squared_error(y_v, predictions)

    mse_ols_results.append(mse)

print(f"MSE from each fold: {mse_ols_results}")

MSE from each fold: [26.125475657135624, 25.69432714173296]


In [9]:
#average MSE from this model
avg_mse = sum(mse_ols_results)/len(mse_ols_results)
print(avg_mse)

25.90990139943429


In [11]:
#test model
#scikitlearn is being annoying, reshape other variables now as well
#run linear regression on the training data now:
model1.fit(X_train, y_train)

y_predicted = model1.predict(X_test)
test_mse = mean_squared_error(y_test, y_predicted)
print(test_mse)

17.12212198895926


In [12]:
slope = model1.coef_[0]
intercept = model1.intercept_

print(slope)
print(intercept)

-0.08574411969315872
11.945383854048716


In [13]:
y_predicted

array([10.48773382,  9.45880438, 11.55953532, 11.64527944, 10.02471557,
       11.38804708, 11.61098179, 10.91645442,  6.0290396 , 11.51666326,
        7.65817787,  7.65817787,  9.88752498,  9.33876262, 11.30230296,
        9.63029262])

##### Diagnostics on model TO BE UPDATED LATER!!!

In [22]:
def least_squares_diagnostics(x,y):
    #adjust this to work with model!
    # do the least squares analysis
    #X = sm.add_constant(x)
    #model = sm.OLS(y,X).fit()
    #calculate ideal y values
    yhat = y_predicted
    #yhat = model.predict(X)
    #calculate influence
    influence = model1.get_influence()
    s_i = influence.resid_studentized_external # standard residuals
    h_ii = influence.hat_matrix_diag # leverages
    h_grid = np.linspace(np.min(h_ii),np.max(h_ii),100)

    #n, p = X.shape
    n, p = X_test.shape
    dof = n - p # degrees of freedom
    l_thr = 2*p/n # leverage threshold
    d_thr = 4.0 /dof # cook's distance threshold
    s_cook = np.sqrt(d_thr * p * (1-h_grid) / h_grid)
#plotting
    fig, ax = plt.subplots(4,1,sharex=False,sharey=False, figsize=(6,15))

    plt.suptitle('Least squares diagnostics, R2: {0:5.4f}'.format(model.rsquared))
    # residual plot
    #ADD THE HORIZONTAL BAR
    ax[0].set_title('Residuals')
    ax[0].scatter(y,y-yhat, alpha =0.5, color = 'black')
    ax[0].hlines(0,np.min(y),np.max(y),linewidth=3, color = 'hotpink')
    ax[0].set_xlabel('sleep onset delay')
    ax[0].set_ylabel('Residuals')

    # q-q plot
    osm, osr = stats.probplot(s_i,dist='t',fit=False,sparams=(n-p-1))
    ax[2].set_title('Q-Q')
    ax[2].scatter(osm,osr, color = 'black', alpha =0.5)
    ax[2].plot(osm,osm, linewidth = 3, color = 'mediumseagreen')
    ax[2].set_xlabel('Theoretical quantile')
    ax[2].set_ylabel('Real quantile')
    
    # scale-location plot
    ax[1].set_title('Scale-location')
    ax[1].scatter(yhat,np.sqrt(abs(s_i)), color = 'black', alpha =0.5)
    ax[1].set_xlabel('Predicted values')
    ax[1].set_ylabel('|Std. Res.|$^{1/2}$')
    
    # residuals vs leverage plot
    ax[3].set_title('Residuals vs leverage')
    ax[3].scatter(h_ii,s_i, alpha =0.5, color = 'black')
    if( np.min(s_cook) < np.max(s_i) or np.max(-s_cook) > np.min(s_i) ):
        plt.plot(h_grid,s_cook, color = 'darkorange', linewidth = 3)
        plt.plot(h_grid,-s_cook, color = 'darkorange', linewidth = 3)
    if( l_thr < 1.1 * np.max(h_ii) ):
        plt.vlines(l_thr,np.min(s_i),np.max(s_i), color = 'royalblue', linewidth = 3)
    if( np.max(s_i) > 2.5 or np.min(s_i) < -2.5 ):
        plt.hlines([-3,3],np.min(h_ii),np.max(h_ii),color = 'royalblue', linewidth = 3)
    ax[3].set_xlabel('Leverage')
    ax[3].set_ylabel('Std. Res.')

    
    plt.tight_layout()
    plt.show()

    return